# 04 COF 描述符：从结构到机器学习特征

模型不能直接理解“孔大”“含氮”“层间滑移”。材料信息需要转换成 numerical features / descriptors。


## 1. COF 特征分层

| 层级 | 例子 | 来源 |
|---|---|---|
| Composition | C/N/O/F fraction | formula / CIF |
| Crystal | a,b,c,angles,volume,density | CIF |
| Pore | PLD,LCD,ASA,void fraction,pore volume | pore software |
| Chemistry | functional groups, RAC/local environment | structure + chemistry tools |
| Learned | graph embedding | GNN |

能生成很多列，不代表都应该进入模型。


In [ ]:
import pandas as pd
example=pd.DataFrame({'COF':['A','B','C'],'density':[0.55,0.78,0.61],'PLD_A':[12,7.5,10.1],'LCD_A':[18,11.2,15.4],'void_fraction':[0.72,0.51,0.65],'N_fraction':[0.08,0.13,0.05]})
example['LCD_PLD_ratio']=example['LCD_A']/example['PLD_A']
display(example)


## 2. Feature engineering
有物理意义地组合已有特征也是特征构造，例如 `LCD/PLD`、heteroatom fraction。复杂不等于更好。


In [ ]:
!pip -q install pymatgen matminer
from pymatgen.core import Composition
from matminer.featurizers.composition import ElementProperty
tmp=pd.DataFrame({'composition':[Composition(x) for x in ['C6H6','C6H4N2','C6H4O2']]})
feat=ElementProperty.from_preset('magpie')
out=feat.featurize_dataframe(tmp,col_id='composition',ignore_errors=True)
print('generated columns =',out.shape[1]-1)
display(out.iloc[:,:8])


## 3. 特征必须匹配 target
CO₂ adsorption、band gap、mechanical stability、diffusion coefficient 不应机械使用完全相同的 feature 集合。

### 完成标准
知道哪些量可直接从 CIF 得到，哪些需要 pore analysis，并能解释为什么 feature selection 必须由科学问题驱动。
